In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

In [2]:
df=pd.read_excel("Wildlife Crime News Database.xlsx",sheet_name="Raw_Articles")

In [3]:
print(df.head(5))

           Date_Collected   Relevance  \
0 2026-01-29 23:07:15.329  ✅ Relevant   
1 2026-01-29 23:07:17.090  ✅ Relevant   
2 2026-01-29 23:07:19.197  ✅ Relevant   
3 2026-01-29 23:07:22.756  ✅ Relevant   
4 2026-01-29 23:07:24.629  ✅ Relevant   

                                               Title  \
0                                        Google News   
1  Palamu Tiger Reserve Seizes 60kg Pangolin Scal...   
2  Poaching alarm near Ranthambhore: Seized bones...   
3  Woman held with 2 kg pangolin scales in Jharkh...   
4  Jamshedpur Forest Officials Eye Kolkata Kingpi...   

                                             Snippet  \
0  Elephant found shot dead near India–Bhutan bor...   
1  Palamu Tiger Reserve Seizes 60kg Pangolin Scal...   
2  Poaching alarm near Ranthambhore: Seized bones...   
3  Woman held with 2 kg pangolin scales in Jharkh...   
4  Jamshedpur Forest Officials Eye Kolkata Kingpi...   

                                                 URL              Source  \
0  

In [4]:
print(df.columns)

Index(['Date_Collected', 'Relevance', 'Title', 'Snippet', 'URL', 'Source',
       'Processed', 'Search_Keyword', 'Duplicate', 'Unnamed: 9',
       'Unnamed: 10'],
      dtype='str')


In [5]:
columns_needed = [
    "Title",
    "Snippet",
    "Relevance"
]

existing_columns = [
    col for col in columns_needed
    if col in df.columns
]

df = df[existing_columns]

print(df.head())

                                               Title  \
0                                        Google News   
1  Palamu Tiger Reserve Seizes 60kg Pangolin Scal...   
2  Poaching alarm near Ranthambhore: Seized bones...   
3  Woman held with 2 kg pangolin scales in Jharkh...   
4  Jamshedpur Forest Officials Eye Kolkata Kingpi...   

                                             Snippet   Relevance  
0  Elephant found shot dead near India–Bhutan bor...  ✅ Relevant  
1  Palamu Tiger Reserve Seizes 60kg Pangolin Scal...  ✅ Relevant  
2  Poaching alarm near Ranthambhore: Seized bones...  ✅ Relevant  
3  Woman held with 2 kg pangolin scales in Jharkh...  ✅ Relevant  
4  Jamshedpur Forest Officials Eye Kolkata Kingpi...  ✅ Relevant  


In [6]:
df.columns

Index(['Title', 'Snippet', 'Relevance'], dtype='str')

In [7]:
df = df.dropna(subset=["Title"])

print(df.shape)

(1103, 3)


In [8]:
df

,Title,Snippet,Relevance
0,Google News,Elephant found shot dead near India–Bhutan bor...,✅ Relevant
1,Palamu Tiger Reserve Seizes 60kg Pangolin Scal...,Palamu Tiger Reserve Seizes 60kg Pangolin Scal...,✅ Relevant
2,Poaching alarm near Ranthambhore: Seized bones...,Poaching alarm near Ranthambhore: Seized bones...,✅ Relevant
3,Woman held with 2 kg pangolin scales in Jharkh...,Woman held with 2 kg pangolin scales in Jharkh...,✅ Relevant
4,Jamshedpur Forest Officials Eye Kolkata Kingpi...,Jamshedpur Forest Officials Eye Kolkata Kingpi...,✅ Relevant
...,...,...,...
1098,சிங்கப்பூரில் வனவிலங்குக் கடத்தல் கும்பல் இல்லை,சிங்கப்பூரில் வனவிலங்குக் கடத்தல் கும்பல் இல்ல...,NaN
1099,ప్రపంచంలో ఎక్కువగా స్మగ్లింగ్ అవుతున్న 10 ప్రా...,ప్రపంచంలో ఎక్కువగా స్మగ్లింగ్ అవుతున్న 10 ప్రా...,NaN
1100,వన్యప్రాణుల జాడకు రోబో… విద్యార్థి ఆశ్రిత్ ప్ర...,వన్యప్రాణుల జాడకు రోబో… విద్యార్థి ఆశ్రిత్ ప్ర...,NaN
1101,ప్రాణం తీసిన మటన్ ముక్క #MahabubabadDistrict #...,ప్రాణం తీసిన మటన్ ముక్క #MahabubabadDistrict #...,NaN


In [9]:
df["label"] = ""

In [10]:
# CELL 9
# USE EXISTING RELEVANCE LABELS

for i, row in df.iterrows():

    relevance = str(
        row["Relevance"]
    ).lower()

    if "relevant" in relevance:

        df.at[i, "label"] = "crime"

    elif "irrelevant" in relevance:

        df.at[i, "label"] = "other"

In [12]:
# CELL 10
# AUTO LABEL REMAINING NULL RECORDS

crime_keywords = [

    "poaching",
    "poacher",
    "trafficking",
    "smuggling",
    "ivory",
    "pangolin",
    "tiger",
    "leopard skin",
    "animal skin",
    "wildlife crime",
    "forest officials",
    "illegal trade",
    "seized",
    "arrested",
    "raid",
    "gang",
    "tusks",
    "deer meat",
    "bird trafficking",
    "exotic animals",
    "snake venom"
]

other_keywords = [

    "awareness",
    "seminar",
    "conservation",
    "tourism",
    "documentary",
    "history",
    "workshop",
    "research",
    "policy",
    "funding",
    "school program",
    "wildlife protection",
    "national park tourism"
]

for i, row in df.iterrows():

    if row["label"] != "":

        continue

    text = row["Title"]

    crime_match = any(

        word in text

        for word in crime_keywords
    )

    other_match = any(

        word in text

        for word in other_keywords
    )

    if crime_match:

        df.at[i, "label"] = "crime"

    elif other_match:

        df.at[i, "label"] = "other"

In [13]:
df.head(50)

,Title,Snippet,Relevance,label
0,Google News,Elephant found shot dead near India–Bhutan bor...,✅ Relevant,crime
1,Palamu Tiger Reserve Seizes 60kg Pangolin Scal...,Palamu Tiger Reserve Seizes 60kg Pangolin Scal...,✅ Relevant,crime
2,Poaching alarm near Ranthambhore: Seized bones...,Poaching alarm near Ranthambhore: Seized bones...,✅ Relevant,crime
3,Woman held with 2 kg pangolin scales in Jharkh...,Woman held with 2 kg pangolin scales in Jharkh...,✅ Relevant,crime
4,Jamshedpur Forest Officials Eye Kolkata Kingpi...,Jamshedpur Forest Officials Eye Kolkata Kingpi...,✅ Relevant,crime
5,Bengaluru Customs foil wildlife smuggling atte...,Bengaluru Customs foil wildlife smuggling atte...,✅ Relevant,crime
6,Passenger Arrested At Mumbai Airport For Smugg...,Passenger Arrested At Mumbai Airport For Smugg...,✅ Relevant,crime
7,"Jamshedpur Wildlife Smuggling Bust: Parrots, C...","Jamshedpur Wildlife Smuggling Bust: Parrots, C...",✅ Relevant,crime
8,Illegal wildlife trade busted from Crawford Ma...,Illegal wildlife trade busted from Crawford Ma...,✅ Relevant,crime
9,Three arrested for illegal wildlife trade in N...,Three arrested for illegal wildlife trade in N...,✅ Relevant,crime


In [14]:
print(

    df["Relevance"]
    .isnull()
    .sum()
)

1058


In [15]:
crime_keywords = [

    "poaching",
    "poacher",
    "trafficking",
    "smuggling",
    "ivory",
    "pangolin",
    "tiger",
    "leopard skin",
    "animal skin",
    "wildlife crime",
    "forest officials",
    "illegal trade",
    "seized",
    "arrested",
    "raid",
    "gang",
    "tusks",
    "deer meat",
    "bird trafficking",
    "exotic animals",
    "snake venom",
    "hunting",
    "rhino horn",
    "wildlife traffickers",
    "elephant tusk"
]

other_keywords = [

    "awareness",
    "seminar",
    "conservation",
    "tourism",
    "documentary",
    "history",
    "workshop",
    "research",
    "policy",
    "funding",
    "school program",
    "wildlife protection",
    "national park tourism",
    "essay",
    "photography",
    "wildlife sanctuary",
    "forest department event"
]

for i, row in df.iterrows():

    if row["label"] != "":

        continue

    text = str(
        row["Title"]
    ).lower()

    crime_score = sum(

        word in text

        for word in crime_keywords
    )

    other_score = sum(

        word in text

        for word in other_keywords
    )

    # -------------------------
    # LABELING
    # -------------------------

    if crime_score >= 1:

        df.at[i, "label"] = "crime"

    elif other_score >= 1:

        df.at[i, "label"] = "other"

    else:

        # UNKNOWN ARTICLES

        df.at[i, "label"] = "other"

In [16]:
df.tail(10)

,Title,Snippet,Relevance,label
1093,দস্যুমুক্ত সুন্দরবন গড়তে টানা অভিযান চলবে: কো...,দস্যুমুক্ত সুন্দরবন গড়তে টানা অভিযান চলবে: কো...,NaN,other
1094,ജഗ്ദൽപൂരിൽ 16 കിലോ ഈനാംപേച്ചി തോടുകൾ പിടികൂടി;...,ജഗ്ദൽപൂരിൽ 16 കിലോ ഈനാംപേച്ചി തോടുകൾ പിടികൂടി;...,NaN,crime
1095,ഛത്തീസ്​ഗഡിൽ ഈനാംപേച്ചിക്കടത്ത്: മൂന്ന് പേർ പി...,ഛത്തീസ്​ഗഡിൽ ഈനാംപേച്ചിക്കടത്ത്: മൂന്ന് പേർ പി...,NaN,other
1096,അമേരിക്ക-ഇറാൻ ചർച്ചകൾ പരാജയം; മധ്യസ്ഥതയ്ക്ക് ത...,അമേരിക്ക-ഇറാൻ ചർച്ചകൾ പരാജയം; മധ്യസ്ഥതയ്ക്ക് ത...,NaN,other
1097,ഛത്തീസ്ഗഢില്‍ വൻ വന്യജീവി വേട്ട: 16 കിലോയിലധിക...,ഛത്തീസ്ഗഢില്‍ വൻ വന്യജീവി വേട്ട: 16 കിലോയിലധിക...,NaN,other
1098,சிங்கப்பூரில் வனவிலங்குக் கடத்தல் கும்பல் இல்லை,சிங்கப்பூரில் வனவிலங்குக் கடத்தல் கும்பல் இல்ல...,NaN,other
1099,ప్రపంచంలో ఎక్కువగా స్మగ్లింగ్ అవుతున్న 10 ప్రా...,ప్రపంచంలో ఎక్కువగా స్మగ్లింగ్ అవుతున్న 10 ప్రా...,NaN,other
1100,వన్యప్రాణుల జాడకు రోబో… విద్యార్థి ఆశ్రిత్ ప్ర...,వన్యప్రాణుల జాడకు రోబో… విద్యార్థి ఆశ్రిత్ ప్ర...,NaN,other
1101,ప్రాణం తీసిన మటన్ ముక్క #MahabubabadDistrict #...,ప్రాణం తీసిన మటన్ ముక్క #MahabubabadDistrict #...,NaN,other
1102,యర్రగొండపాలెంలో కార్డెన్ సర్చ్,యర్రగొండపాలెంలో కార్డెన్ సర్చ్ Lokal Telugu,NaN,other


In [17]:
from deep_translator import (
    GoogleTranslator
)

In [18]:
def translate_text(text):

    try:

        return GoogleTranslator(

            source='auto',

            target='en'

        ).translate(text)

    except:

        return text

In [19]:
from deep_translator import (
    GoogleTranslator
)

import re

In [20]:
def is_english(text):

    return bool(

        re.match(

            r'^[a-zA-Z0-9\s\W]+$',

            str(text)
        )
    )

In [21]:
translated_titles = []

for i, title in enumerate(df["Title"]):

    title = str(title)

    try:

        # -------------------------
        # SKIP ENGLISH
        # -------------------------

        if is_english(title):

            translated_titles.append(
                title
            )

        else:

            translated = GoogleTranslator(

                source='auto',

                target='en'

            ).translate(title)

            translated_titles.append(
                translated
            )

        # -------------------------
        # PROGRESS
        # -------------------------

        if i % 50 == 0:

            print(
                f"Processed {i}"
            )

    except:

        translated_titles.append(
            title
        )

df["translated_title"] = (
    translated_titles
)

Processed 0
Processed 50
Processed 100
Processed 150
Processed 200
Processed 250
Processed 300
Processed 350
Processed 400
Processed 450
Processed 500
Processed 550
Processed 600
Processed 650
Processed 700
Processed 750
Processed 800
Processed 850
Processed 900
Processed 950
Processed 1000
Processed 1050
Processed 1100


In [26]:
df.head(50)

,Title,Snippet,Relevance,label,translated_title
0,Google News,Elephant found shot dead near India–Bhutan bor...,✅ Relevant,crime,Google News
1,Palamu Tiger Reserve Seizes 60kg Pangolin Scal...,Palamu Tiger Reserve Seizes 60kg Pangolin Scal...,✅ Relevant,crime,Palamu Tiger Reserve Seizes 60kg Pangolin Scal...
2,Poaching alarm near Ranthambhore: Seized bones...,Poaching alarm near Ranthambhore: Seized bones...,✅ Relevant,crime,Poaching alarm near Ranthambhore: Seized bones...
3,Woman held with 2 kg pangolin scales in Jharkh...,Woman held with 2 kg pangolin scales in Jharkh...,✅ Relevant,crime,Woman held with 2 kg pangolin scales in Jharkh...
4,Jamshedpur Forest Officials Eye Kolkata Kingpi...,Jamshedpur Forest Officials Eye Kolkata Kingpi...,✅ Relevant,crime,Jamshedpur Forest Officials Eye Kolkata Kingpi...
5,Bengaluru Customs foil wildlife smuggling atte...,Bengaluru Customs foil wildlife smuggling atte...,✅ Relevant,crime,Bengaluru Customs foil wildlife smuggling atte...
6,Passenger Arrested At Mumbai Airport For Smugg...,Passenger Arrested At Mumbai Airport For Smugg...,✅ Relevant,crime,Passenger Arrested At Mumbai Airport For Smugg...
7,"Jamshedpur Wildlife Smuggling Bust: Parrots, C...","Jamshedpur Wildlife Smuggling Bust: Parrots, C...",✅ Relevant,crime,"Jamshedpur Wildlife Smuggling Bust: Parrots, C..."
8,Illegal wildlife trade busted from Crawford Ma...,Illegal wildlife trade busted from Crawford Ma...,✅ Relevant,crime,Illegal wildlife trade busted from Crawford Ma...
9,Three arrested for illegal wildlife trade in N...,Three arrested for illegal wildlife trade in N...,✅ Relevant,crime,Three arrested for illegal wildlife trade in N...


In [22]:
X = df["translated_title"]

y = df["label"]

In [23]:
X_train, X_test, y_train, y_test = (

    train_test_split(

        X,
        y,

        test_size=0.2,

        random_state=42,

        stratify=y
    )
)

In [24]:
print(len(X_train))
print(len(X_test))

882
221


In [25]:
model = Pipeline([

    (

        "tfidf",

        TfidfVectorizer(

            stop_words="english",

            ngram_range=(1,3),

            max_features=20000,

            min_df=2
        )
    ),

    (

        "clf",

        LogisticRegression(

            max_iter=5000
        )
    )
])

In [26]:
print("TRAINING MODEL...")

model.fit(
    X_train,
    y_train
)

print("MODEL TRAINED")

TRAINING MODEL...
MODEL TRAINED


In [27]:
predictions = model.predict(
    X_test
)

In [28]:
from sklearn.metrics import (

    accuracy_score,

    classification_report
)

accuracy = accuracy_score(

    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 89.14 %


In [29]:
print(

    classification_report(

        y_test,
        predictions
    )
)

              precision    recall  f1-score   support

       crime       0.92      0.80      0.86        91
       other       0.87      0.95      0.91       130

    accuracy                           0.89       221
   macro avg       0.90      0.88      0.89       221
weighted avg       0.89      0.89      0.89       221



In [30]:
import joblib

joblib.dump(

    model,

    "../models/wildlife_crime_model.pkl"
)

print("✅ MODEL SAVED")

✅ MODEL SAVED
